# Brent-Dekker Method

Brent's method (also known as the Brent-Dekker method) is a root-finding algorithm that combines the bisection method, the secant method, and inverse quadratic interpolation. It has the reliability of bisection but can be as fast as the secant method or inverse quadratic interpolation.

**Key ideas:**

1.  **Guaranteed Convergence:** Like the bisection method, it requires an initial interval `[a, b]` where `f(a)` and `f(b)` have opposite signs (i.e., `f(a) * f(b) < 0`), guaranteeing that a root exists within the interval.
2.  **Speed of Secant/IQI:** It attempts to use faster methods (secant method or inverse quadratic interpolation) to propose new points.
3.  **Fallback to Bisection:** If the faster methods fail to produce a satisfactory new point (e.g., if it falls outside the bracket or convergence is slow), it falls back to a bisection step to maintain convergence guarantees and keep the interval bracketing the root small.

This combination makes Brent's method a very robust and efficient root-finding algorithm for functions where a bracketed root is known.
***
## Brent's Algorithm
This version of Brent's method (like `brentq`) aims for robust root-finding by cleverly combining techniques. We'll simplify the explanation, focusing on bisection and the secant method with a new error tolerance.

Given:
*   A function $f(x)$.
*   An initial interval $[a, b]$ where $f(a)$ and $f(b)$ have opposite signs (a root is guaranteed inside).
*   An error tolerance $e = 0.001$.

Here are the simplified steps:

1.  **Initialization:**
    *   Calculate $fa = f(a)$ and $fb = f(b)$.
    *   Make sure $fa \cdot fb < 0$. If not, the interval is invalid.
    *   For internal consistency, if $|fa| < |fb|$, swap $a$ with $b$ and $fa$ with $fb$.
    *   Keep track of previous points: $c = a$, $fc = fa$. A previous `d` will also be used.

2.  **Loop for Root Finding:** Repeat these steps until the root is found:

    *   **Check for Convergence:** If $|fa - fb| \le e$, then we consider `b` to be an acceptable root. Stop and return `b`.

    *   **Propose a New Point ($s$):**
        *   The algorithm tries to use the **Secant Method** to predict a new point `s`. This method uses a line between two points on the function to estimate where the function crosses zero.
        *   The formula used is typically: $s = b - fb \cdot (b - a) / (fb - fa)$.

    *   **Safeguard (Fallback to Bisection):**
        *   To ensure stability, the proposed point $s$ is checked. If $s$ is not a "good" estimate (e.g., if it falls outside the current bracket $[a, b]$ or doesn't promise good progress), the algorithm **falls back to Bisection**.
        *   In Bisection, $s$ is simply the midpoint of the current interval: $s = (a + b) / 2$.

    *   **Update the Interval:**
        *   Calculate $fs = f(s)$.
        *   Adjust the interval $[a, b]$ to bracket the root.
        *   If $fa \cdot fs < 0$, the root is in $[a, s]$. So, update $b = s$ and $fb = fs$.
        *   Otherwise (if $fb \cdot fs < 0$), the root is in $[s, b]$. So, update $a = s$ and $fa = fs$.
        *   Always ensure that the point $a$ has the larger absolute function value (if $|fa| < |fb|$, swap $a, b$ and $fa, fb$) for better performance of the secant method.

3.  **Result:** Once converged, `b` is the approximated root. If the loop limit is reached, it indicates no convergence.
***

In [1]:
from math import log

def brent_dekker(f, a, b, tol=0.001, max_iter=100):
    """
    Finds a root of a function using Brent's method.

    Args:
        f (function): The function for which to find the root.
        a (float): One end of the initial bracketing interval.
        b (float): The other end of the initial bracketing interval.
        tol (float): The tolerance for convergence (interpreted as |f(a) - f(b)| <= tol).
        max_iter (int): The maximum number of iterations.

    Returns:
        float: The approximated root.
        None: If the method does not converge within max_iter or initial bracket is invalid.
    """

    fa = f(a)
    fb = f(b)

    if fa * fb >= 0:
        print("Function has the same sign at interval endpoints a and b.")
        print("Brent's method requires f(a) and f(b) to have opposite signs.")
        return None

    # Ensure f(a) has smaller absolute value for some internal logic (original Brent's preference)
    if abs(fa) < abs(fb):
        a, b = b, a
        fa, fb = fb, fa

    c = a
    fc = fa
    d = c
    mflag = True
    iter_count = 0

    while iter_count < max_iter:
        iter_count += 1

        # NEW CONVERGENCE CONDITION as per user's request
        if abs(fa - fb) <= tol:
            print(f"Converged in {iter_count} iterations (based on |f(a) - f(b)| <= {tol}).")
            # The root is considered to be 'b' as it's the point with the smaller |f(x)| initially
            return b

        # Inverse Quadratic Interpolation or Secant Method
        # (Retained for Brent's method robustness, explanation simplified in markdown)
        if fa != fc and fb != fc:
            # Inverse Quadratic Interpolation
            s = (a * fb * fc / ((fa - fb) * (fa - fc)) +
                 b * fa * fc / ((fb - fa) * (fb - fc)) +
                 c * fa * fb / ((fc - fa) * (fc - fb)))
        else:
            # Secant Method
            s = b - fb * (b - a) / (fb - fa)

        # Check if proposed 's' is acceptable (Brent's safeguard conditions)
        # Note: The 'tol' here is now the user-defined tolerance (0.001)
        # for function value difference, which is applied consistently.
        condition1 = (s < (3*a + b) / 4 and s < b) or (s > (3*a + b) / 4 and s > b)
        condition2 = mflag and (abs(s - b) >= abs(b - c) / 2)
        condition3 = (not mflag) and (abs(s - b) >= abs(c - d) / 2)
        condition4 = mflag and (abs(b - c) < tol) # Using user's 'tol' here for interval width check
        condition5 = (not mflag) and (abs(c - d) < tol) # Using user's 'tol' here for interval width check

        if condition1 or condition2 or condition3 or condition4 or condition5:
            s = (a + b) / 2  # Bisection step as fallback
            mflag = True
        else:
            mflag = False

        fs = f(s)
        d = c
        c = b
        fc = fb

        if fa * fs < 0:
            b = s
            fb = fs
        else:
            a = s
            fa = fs

        # Maintain `b` as the point with the smaller absolute function value
        if abs(fa) < abs(fb):
            a, b = b, a
            fa, fb = fb, fa

    print("Maximum number of iterations reached.")
    return b

## Inverse Quadratic Interpolation (IQI)

Inverse Quadratic Interpolation is a numerical method used to find roots of a function, often employed as an accelerated step within more robust algorithms like Brent's method.

The core idea is:

1.  **Inverse Function:** Instead of interpolating the function $y = f(x)$ directly, IQI interpolates the inverse function $x = f^{-1}(y)$.
2.  **Quadratic Fit:** It uses three distinct points $(x_0, y_0)$, $(x_1, y_1)$, and $(x_2, y_2)$ (where $y_i = f(x_i)$) to fit a quadratic polynomial to the inverse function. That is, it finds a quadratic approximation for $x$ as a function of $y$.
3.  **Root Estimate:** Once this quadratic is found, the next estimate for the root is obtained by setting $y=0$ in the quadratic approximation. This effectively finds the $x$-intercept of the inverse function, which corresponds to the root of the original function $f(x)$.

**Why use it?**

*   **Faster Convergence:** When the function is well-behaved near the root, IQI can converge much faster than linear methods like the secant method or bisection.
*   **Efficiency:** It uses more information (three points instead of two) to make a better-informed guess about the root's location.

However, it's not always stable on its own, which is why it's often safeguarded by methods like bisection, as seen in Brent's method.
***
## Application of Brent-Dekker Method
Let's apply Brent-Dekker's method to find the root of $f(x) = x + \ln(x)$. We need an initial interval `[a, b]` where $f(a)$ and $f(b)$ have opposite signs.<br/>For $f(x) = x + \ln(x)$:

*   $f(0.1) = 0.1 + \ln(0.1) \approx 0.1 - 2.3 = -2.2$
*   $f(1) = 1 + \ln(1) = 1 + 0 = 1$

So, the interval `[0.1, 1]` is a valid bracketing interval.

In [2]:
# Define the function
f = lambda x: x + log(x)

# Initial bracketing interval
a = 0.1
b = 1.0

# Find the root using Brent-Dekker method
root_brent_dekker = brent_dekker(f, a, b)

if root_brent_dekker is not None:
    print(f"The approximated root using Brent-Dekker method is: {root_brent_dekker:.6f}")
    print(f"Function value at the root: f({root_brent_dekker:.6f}) = {f(root_brent_dekker):.3f}")

Converged in 4 iterations (based on |f(a) - f(b)| <= 0.001).
The approximated root using Brent-Dekker method is: 0.567163
Function value at the root: f(0.567163) = 0.000


***